In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.2 Pre-normalization

> 🎯 **Goal:** See where normalization sits in a block and why putting it *before*
> each sublayer keeps deep stacks stable.

**Why this matters.** In l4.1 the residual stream carries a running sum, and sums
can drift to large or strange scales as they grow across layers. Pre-norm is the
tidy-up step that hands each sublayer a well-behaved input no matter how big the
stream has gotten. It is the difference between a transformer that trains smoothly
and one that needs fragile learning-rate warmup and careful initialization.

**The intuition.** Before every step, tidy your workspace. Whatever mess has piled
up on the conveyor belt, you take a clean *copy*, straighten it out to a known
scale, and hand that tidy copy to the sublayer. The belt itself (the residual
stream) is never disturbed, only the copy the sublayer reads. So the stream stays
free to grow, while each sublayer always sees neat, normalized inputs.

**The mechanics.** PocketLM computes `x + attn(norm(x))` and `x + mlp(norm(x))`:
the norm wraps the *input* to the sublayer, not the output. This is **pre-norm**.
The older **post-norm** design (`norm(x + attn(x))`) normalizes after the add and
sits directly on the residual path, which makes very deep stacks harder to train.
The normalizer here is **LayerNorm**: for each position's vector it subtracts the
mean and divides by the standard deviation (so the vector has mean 0 and variance
1), then applies a learned scale and shift (the **affine** parameters) so the model
can undo the normalization where it helps. 'Per position' is the key detail: every
token is normalized on its own, using only its own feature values.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
ln = nn.LayerNorm(dim)
x = torch.randn(1, 4, dim) * 5 + 3   # arbitrary scale and shift
normed = ln(x)
row = normed[0, 0]
print("each position is normalized: mean", round(row.mean().item(), 6),
      "std", round(row.std(unbiased=False).item(), 4))

**What just happened.** We took an input deliberately scaled and shifted off-center
(`* 5 + 3`) and ran it through `LayerNorm`. The print shows the result for one
position: mean essentially 0 and std essentially 1. The arbitrary scale and shift
we injected got cancelled, the sublayer downstream would see a clean, standardized
vector. The two asserts pin this down numerically: mean within `1e-5` of zero and
std within `1e-4` of one.

(Why std looks like 1.0 here even before the learned affine: a freshly built
`nn.LayerNorm` starts with weight 1 and bias 0, so the affine is the identity
until training moves it.)

⚠️ **Common pitfalls**
- Confusing LayerNorm with BatchNorm. LayerNorm normalizes across the *features*
  of one token and never looks at other tokens, so it behaves the same whether
  your batch has 1 sequence or 1000. BatchNorm mixes across the batch and is a
  poor fit for sequence models.
- Putting the norm on the residual path (post-norm) by habit. For deep PocketLM
  stacks, pre-norm is the stable choice.
- Using `unbiased=True` when checking the std and being surprised it is not exactly
  1.0. LayerNorm divides by the *biased* (population) std, which is why the check
  uses `unbiased=False`.

🏋️ **Try it yourself.** Change `* 5 + 3` to a wilder scale like `* 100 - 50` and
rerun. The normalized mean and std barely move, that scale-invariance is the whole
point. Then set `ln = nn.LayerNorm(dim, elementwise_affine=False)` and confirm the
output is identical here (because the default affine starts as the identity).

In [ ]:
assert abs(row.mean().item()) < 1e-5
assert abs(row.std(unbiased=False).item() - 1.0) < 1e-4